In [1]:
import random
trips = [
    {
        "name": "Trip 1",
        "arrivals": [5, 10, 15],
        "earliest": 16,
        "latest": 25,
        "duration": 20
    },
    {
        "name": "Trip 2",
        "arrivals": [20, 25, 30],
        "earliest": 31,
        "latest": 45,
        "duration": 25
    },
    {
        "name": "Trip 3",
        "arrivals": [35, 40, 45],
        "earliest": 46,
        "latest": 60,
        "duration": 20
    },
    {
        "name": "Trip 4",
        "arrivals": [50, 55, 60],
        "earliest": 61,
        "latest": 75,
        "duration": 25
    }
]
vehicle_cost = [10, 12, 15]
TURNAROUND = 5
POPULATION_SIZE = 50
GENERATIONS = 200
MUTATION_RATE = 0.15
WAIT_WEIGHT = 1.0
COST_WEIGHT = 0.2
PENALTY = 10000
def create_schedule():
    schedule = []
    for trip in trips:
        departure = random.randint(
            trip["earliest"], trip["latest"]
        )
        vehicle = random.randrange(len(vehicle_cost))
        schedule.append((departure, vehicle))
    return schedule

def objective(schedule):
    waiting_time = 0
    operating_cost = 0
    penalty = 0
    for i, (departure, vehicle) in enumerate(schedule):
        trip = trips[i]

        waiting_time += sum(
            departure - arrival
            for arrival in trip["arrivals"]
        )
        operating_cost += (
            trip["duration"] * vehicle_cost[vehicle]
        )
        if not (
            trip["earliest"] <= departure <= trip["latest"]
        ):
            penalty += PENALTY

    for i in range(len(trips)):
        for j in range(i + 1, len(trips)):
            dep1, vehicle1 = schedule[i]
            dep2, vehicle2 = schedule[j]

            if vehicle1 == vehicle2:
                end1 = dep1 + trips[i]["duration"] + TURNAROUND
                end2 = dep2 + trips[j]["duration"] + TURNAROUND
                if dep1 < end2 and dep2 < end1:
                    penalty += PENALTY
    return (
        WAIT_WEIGHT * waiting_time
        + COST_WEIGHT * operating_cost
        + penalty
    )


def select(population):
    contestants = random.sample(population, 3)

    return min(contestants, key=objective)
def crossover(parent1, parent2):
    point = random.randint(1, len(trips) - 1)
    child = parent1[:point] + parent2[point:]
    return child
def mutate(schedule):
    child = schedule.copy()
    for i, trip in enumerate(trips):
        if random.random() < MUTATION_RATE:
            departure, vehicle = child[i]
            if random.random() < 0.5:
                departure = random.randint(
                    trip["earliest"], trip["latest"]
                )
            else:
                vehicle = random.randrange(len(vehicle_cost))

            child[i] = (departure, vehicle)
    return child
def genetic_algorithm():
    population = [
        create_schedule()
        for _ in range(POPULATION_SIZE)
    ]
    for generation in range(GENERATIONS):
        population.sort(key=objective)
        new_population = population[:2]

        while len(new_population) < POPULATION_SIZE:
            parent1 = select(population)
            parent2 = select(population)

            child = crossover(parent1, parent2)
            child = mutate(child)

            new_population.append(child)

        population = new_population
    return min(population, key=objective)
best = genetic_algorithm()
total_wait = 0
total_cost = 0
print("OPTIMIZED BUS SCHEDULE")
print("-" * 50)
for i, (departure, vehicle) in enumerate(best):
    trip = trips[i]
    wait = sum(
        departure - arrival
        for arrival in trip["arrivals"]
    )
    cost = trip["duration"] * vehicle_cost[vehicle]

    total_wait += wait
    total_cost += cost
    print(
        f"{trip['name']}: "
        f"Departure = {departure} min, "
        f"Vehicle = {vehicle + 1}, "
        f"Waiting = {wait} passenger-min, "
        f"Cost = {cost}"
    )
print("-" * 50)
print("Total passenger waiting time:", total_wait,
      "passenger-minutes")
print("Total operating cost:", total_cost)
print("Objective value:", objective(best))





OPTIMIZED BUS SCHEDULE
--------------------------------------------------
Trip 1: Departure = 16 min, Vehicle = 1, Waiting = 18 passenger-min, Cost = 200
Trip 2: Departure = 31 min, Vehicle = 2, Waiting = 18 passenger-min, Cost = 300
Trip 3: Departure = 46 min, Vehicle = 1, Waiting = 18 passenger-min, Cost = 200
Trip 4: Departure = 61 min, Vehicle = 2, Waiting = 18 passenger-min, Cost = 300
--------------------------------------------------
Total passenger waiting time: 72 passenger-minutes
Total operating cost: 1000
Objective value: 272.0
